# MultiAgent Architecture Generation Experiment

## Objetivo
Gerar arquiteturas de microsserviços usando um experimento multiagente com CrewAI e LangChain.

### Arquitetura do Experimento
- **Agente 1 (Arquiteto A):** Gera proposta independente de decomposição
- **Agente 2 (Arquiteto B):** Gera segunda proposta independente
- **Agente 3 (Validador):** Compara propostas e gera arquitetura consolidada
- **Avaliação:** Comparação com referência usando Precision, Recall, F1

### Requisitos
- Casos de estudo: Bookstore e PetClinic
- Entrada: Requisitos textuais em `../../../Artefatos_Reproducao/Systems Artifacts/`
- Referência: Arquivos `.dot` em `../../../Artefatos_Reproducao/Dot Files/`

---

## Fase 1: Setup & Validação do Ambiente
Este notebook prepara o ambiente sem ainda implementar agentes.

## 1. Importações Necessárias

In [ ]:
# ============================================
# System & Environment
# ============================================
import os
import sys
from pathlib import Path

# ============================================
# Configuration & API Keys
# ============================================
from dotenv import load_dotenv

# ============================================
# LangChain with Google Gemini
# ============================================
from crewai import LLM
from langchain_core.messages import HumanMessage, SystemMessage

# CrewAI
from crewai import Agent, Task, Crew

# ============================================
# Data Processing
# ============================================
import pandas as pd
import json
from typing import Dict, List, Optional

# ============================================
# Utilities
# ============================================
import warnings
warnings.filterwarnings('ignore')

print("✓ Todas as importações carregadas com sucesso")

## 2. Configuração do Ambiente (dotenv)

In [ ]:
# ============================================
# Carregar variáveis de ambiente
# ============================================

# Localizar .env no diretório pai do notebook
notebook_dir = Path('.').absolute()
env_file = notebook_dir.parent / '.env'

print(f"Procurando .env em: {env_file}")

if env_file.exists():
    load_dotenv(env_file)
    print(f"✓ Arquivo .env carregado de: {env_file}")
else:
    print(f"⚠ Arquivo .env não encontrado em {env_file}")
    print(f"  Usando variáveis de ambiente do sistema")
    load_dotenv()  # Tenta carregar do ambiente global

# ============================================
# Verificar configurações críticas
# ============================================

google_api_key = os.getenv("GOOGLE_API_KEY")
google_model = os.getenv("GOOGLE_MODEL", "gemini-1.5-pro")  # Default: gemini-1.5-pro
debug_mode = os.getenv("LANGCHAIN_DEBUG", "False").lower() == "true"

print(f"\n=== Configuração Atual ===")
print(f"Provider: Google Gemini")
print(f"Model: {google_model}")
print(f"API Key: {'✓ Configurada' if google_api_key else '✗ NÃO CONFIGURADA'}")
print(f"Debug Mode: {debug_mode}")

if not google_api_key:
    print("\n⚠ AVISO: GOOGLE_API_KEY não foi encontrada!")
    print("  1. Copie .env.example para .env")
    print("  2. Adicione sua chave: GOOGLE_API_KEY=your-gemini-key...")
    print("  3. Execute novamente")

## 3. Inicialização do Modelo Google Gemini (LangChain)

In [ ]:
# ============================================
# Inicializar ChatGoogleGenerativeAI via LangChain
# ============================================

try:
    # Configurar o modelo Gemini
    llm = ChatGoogleGenerativeAI(
        model=google_model,            # "gemini-1.5-pro" configurado no .env
        temperature=0.3,               # Determinístico (reproduzível)
        api_key=google_api_key,
        max_output_tokens=2000,        # Limite de tokens para resposta
    )
    
    print("✓ Modelo Google Gemini inicializado com sucesso via LangChain")
    print(f"  - Modelo: {google_model}")
    print(f"  - Temperatura: 0.3 (determinístico)")
    print(f"  - Max Output Tokens: 2000")
    
except Exception as e:
    print(f"✗ Erro ao inicializar modelo: {str(e)}")
    print("\n  Possíveis causas:")
    print("  1. GOOGLE_API_KEY não configurada")
    print("  2. Credenciais inválidas")
    print("  3. Sem conexão à internet")
    llm = None

## 4. Teste de Conexão com o Modelo

In [ ]:
# ============================================
# Teste simples: verificar comunicação com o modelo
# ============================================

if llm is not None:
    print("Enviando mensagem de teste para o modelo Gemini...\n")
    
    try:
        # Mensagem de teste
        test_prompt = "Você é um assistente de arquitetura de software. Confirme que está funcionando respondendo apenas: 'Pronto para gerar arquiteturas de microsserviços!'"
        
        # Fazer chamada ao modelo
        response = llm.invoke([
            SystemMessage(content="Você é um assistente de arquitetura de software."),
            HumanMessage(content=test_prompt)
        ])
        
        # Exibir resposta
        print("=== RESPOSTA DO MODELO ===")
        print(response.content)
        print("\n✓ Conexão com modelo Google Gemini validada com sucesso!")
        print("\n📋 Próximas etapas:")
        print("  1. Definir estrutura de agentes (Agent A, Agent B, Validator)")
        print("  2. Implementar tasks para geração de arquitetura")
        print("  3. Criar pipeline de consolidação e avaliação")
        
    except Exception as e:
        print(f"✗ Erro na chamada ao modelo: {str(e)}")
        print("\n  Verifique:")
        print("  - API Key válida e com créditos")
        print("  - Conexão à internet ativa")
        print("  - Limites de taxa (rate limits) não excedidos")
else:
    print("⚠ Modelo não foi inicializado. Configure GOOGLE_API_KEY e execute novamente.")

## 5. Informações do Ambiente

Próximas células serão utilizadas para:  
- **Fase 2:** Definição de agentes (Arquiteto A, Arquiteto B, Validador)  
- **Fase 3:** Implementação de tasks (geração, comparação, consolidação)  
- **Fase 4:** Orquestração com Crew  
- **Fase 5:** Avaliação (Precision, Recall, F1)  

In [ ]:
# ============================================
# Exibir informações do projeto
# ============================================

print("\n" + "="*60)
print("MULTIAGENT ARCHITECTURE GENERATION - AMBIENTE PREPARADO")
print("="*60)

print(f"\n📁 Estrutura do Projeto:")
print(f"  Notebook: {Path('.').absolute()}")
print(f"  Requisitos entrada: ../../../Artefatos_Reproducao/Systems Artifacts/")
print(f"  Referência: ../../../Artefatos_Reproducao/Dot Files/")

print(f"\n📦 Dependências Instaladas:")
print(f"  ✓ LangChain com ChatGoogleGenerativeAI")
print(f"  ✓ CrewAI (para orquestração de agentes)")
print(f"  ✓ Google Gemini API (modelo: {google_model})")

print(f"\n✅ Status: PRONTO para próxima fase")
print(f"   Aguardando implementação de agentes e tasks...\n")

## Fase 2: Implementação do Primeiro Agente
Agora implementamos o **Software Architect Agent** - nosso primeiro agente do experimento multiagente.

### 6.1 Descomentação de CrewAI

In [ ]:
# ============================================
# CrewAI Imports (Ativados para Fase 2)
# ============================================

from crewai import Agent, Task, Crew
from crewai_tools import tool

print("✓ CrewAI imports ativados com sucesso")

### 6.2 Definição do Primeiro Agente: Software Architect Agent

**Especialização:** Microsserviços, Domain Driven Design (DDD), Decomposição baseada em responsabilidades

Este agente:
- Recebe requisitos textuais do sistema
- Propõe microsserviços baseado em bounded contexts (DDD)
- Define responsabilidades de cada serviço
- Mapeia comunicações necessárias

In [ ]:
# ============================================
# Agente 1: Software Architect Agent
# ============================================

software_architect_agent = Agent(
    role="Software Architect",
    goal="""Propor uma arquitetura de microsserviços baseada em requisitos textuais.
    
    Para cada microsserviço, você deve:
    1. Identificar um nome descritivo baseado em responsabilidades de negócio
    2. Listar as responsabilidades principais
    3. Especificar quais outros serviços ele se comunica
    
    Use princípios de Domain Driven Design (DDD) e bounded contexts.""",
    
    backstory="""Você é um arquiteto de software sênior especializado em microsserviços.
    
    Sua experiência inclui:
    - Decomposição de monólitos usando Domain Driven Design
    - Identificação de bounded contexts
    - Design de comunicação entre serviços
    - Análise de responsabilidades de negócio
    
    Você pensa criteriosamente sobre a estrutura de um sistema,
    buscando separação clara de responsabilidades e coesão máxima.""",
    
    llm=llm,  # Usar modelo Google Gemini configurado anteriormente
    verbose=True,  # Exibir pensamentos do agente
    memory=False,  # Manter histórico de contexto
)

print("✓ Software Architect Agent criado com sucesso")
print(f"  Role: {software_architect_agent.role}")
print(f"  Model: {google_model}")
print(f"  Verbose: {software_architect_agent.verbose}")

### 6.3 Definição da Task: Generate Microservices Architecture

A task define o trabalho específico que o agente deve executar.

**Entrada:** Requisitos textuais do sistema (ex.: PetClinic, Bookstore)
**Saída:** Lista de microsserviços com responsabilidades e comunicações

In [ ]:
# ============================================
# Task 1: Generate Microservices Architecture
# ============================================

generate_architecture_task = Task(
    description="""Analyze the following system requirements and propose a microservices architecture.
    
    Your output MUST be EXACTLY in this CSV format (no markdown, no explanations):
    
    Microservice,Responsibilities,Communicates With
    Service Name,responsibility1;responsibility2;responsibility3,Service1;Service2;Service3
    
    Rules:
    - One row per microservice
    - Separate multiple items with semicolons (;)
    - NO markdown formatting, NO headers, NO extra text
    - List services in a logical order
    - Consider DDD bounded contexts
    
    System Requirements:
    {requirements}
    """,
    
    expected_output="CSV format with microservices, their responsibilities, and communications",
    agent=software_architect_agent,
)

print("✓ Task 'Generate Microservices Architecture' criada com sucesso")
print(f"  Agent: {generate_architecture_task.agent.role}")
print(f"  Expected Output Format: CSV")

### 6.4 Teste do Agente: Spring PetClinic

Executamos o primeiro agente com os requisitos do Spring PetClinic.

**Sistema:** Spring PetClinic - Sistema de clínica veterinária com gestão de clientes, pets, veterinários e visitas.

**Esperado:** Arquitetura com microsserviços como Customers Service, Vets Service, Visits Service, etc.

In [ ]:
# ============================================
# Requisitos do Spring PetClinic
# ============================================

petclinic_requirements = """
Spring PetClinic Microservices - Requirements

1. Manage Clients:
   - Add, update, and delete client information.
   - View detailed information about existing clients.

2. Manage Pets:
   - Add, update, and delete pet information.
   - Link pets to their respective owners.

3. Manage Visits:
   - Register new visits for pets.
   - View visit history for a specific pet.

4. Manage Veterinarians:
   - Add, update, and delete vet information.
   - View specialties for each veterinarian.

5. Service Discovery:
   - Register and locate services using Eureka.

6. Centralized Configuration:
   - Manage service configurations through a Config Server.

7. API Gateway:
   - Route client requests to appropriate backend services.

8. Monitoring and Administration:
   - Monitor services and applications using Admin Server.
"""

print("✓ Requisitos PetClinic carregados")
print(f"  Tamanho: {len(petclinic_requirements)} caracteres")
print(f"  8 áreas de responsabilidade identificadas")

In [ ]:
# ============================================
# Executar Agente 1 com PetClinic
# ============================================

print("="*60)
print("EXECUTANDO AGENTE 1: SOFTWARE ARCHITECT")
print("="*60)
print(f"\nSistema: Spring PetClinic")
print(f"Agente: {software_architect_agent.role}")
print(f"Task: Generate Microservices Architecture\n")

try:
    # Criar Crew com o agente e task
    crew = Crew(
        agents=[software_architect_agent],
        tasks=[generate_architecture_task],
        verbose=True,
    )
    
    # Executar o crew com os requisitos do PetClinic
    petclinic_architecture_output = crew.kickoff(
        inputs={"requirements": petclinic_requirements}
    )
    
    print("\n" + "="*60)
    print("RESULTADO DA ARQUITETURA GERADA")
    print("="*60)
    print(petclinic_architecture_output)
    print("\n✓ Agente executado com sucesso!")
    
except Exception as e:
    print(f"✗ Erro ao executar agente: {str(e)}")
    print(f"\nPossíveis causas:")
    print(f"  1. API Key inválida ou sem créditos")
    print(f"  2. Quota do Google Gemini excedida")
    print(f"  3. Rate limit excedido")
    print(f"  4. Erro de conexão com Google Gemini")
    petclinic_architecture_output = None

### 6.5 Status da Implementação

✅ **Fase 2: Primeiro Agente**
- [x] Software Architect Agent criado
- [x] Task "Generate Microservices Architecture" definida
- [x] Teste executado com PetClinic

📋 **Próximas Fases:**
- [ ] **Fase 3:** Criar segundo agente (Architect B) com prompt ligeiramente diferente
- [ ] **Fase 4:** Criar agente Validador que compara arquiteturas A e B
- [ ] **Fase 5:** Implementar comparação com referência (Precision, Recall, F1)
- [ ] **Fase 6:** Executar experimento completo com Bookstore e PetClinic

### Notas Importantes

1. **CrewAI vs LangChain:** Usamos `Crew` para orquestração, com modelo base LangChain's `ChatGoogleGenerativeAI` (Gemini)
2. **Temperature:** Mantida em 0.3 para resultados determinísticos e reproduzíveis
3. **Formato de Saída:** CSV é essencial para parsing automático
4. **Próxima Iteração:** O segundo agente receberá prompts ligeiramente diferentes para gerar propostas alternativas

In [ ]:
# ============================================
# Resumo da Implementação - Fase 2
# ============================================

print("\n" + "█"*70)
print("RESUMO: PRIMEIRO AGENTE IMPLEMENTADO COM SUCESSO")
print("█"*70 + "\n")

print("📋 AGENTE CRIADO:")
print(f"  ├─ Nome: Software Architect Agent")
print(f"  ├─ Papel: Especialista em Microsserviços & DDD")
print(f"  ├─ Modelo: {google_model}")
print(f"  └─ Temperatura: 0.3 (determinístico)\n")

print("📝 TASK DEFINIDA:")
print(f"  ├─ Nome: Generate Microservices Architecture")
print(f"  ├─ Entrada: Requisitos textuais do sistema")
print(f"  └─ Saída: CSV com microsserviços, responsabilidades e comunicações\n")

print("✅ TESTE EXECUTADO:")
print(f"  ├─ Sistema: Spring PetClinic")
print(f"  ├─ Status: {'✓ Sucesso' if petclinic_architecture_output else '✗ Falha (verifique API Key)'}")
print(f"  └─ Resultado: Arquitetura gerada em formato CSV\n")

print("🎯 PRÓXIMO PASSO:")
print(f"  Criar segundo agente (Architect B) com variação de prompts")
print(f"  para gerar propostas alternativas independentes\n")

print("█"*70)

---
## Fase 3: Segundo Agente (Architect B)

Criamos o segundo agente com prompts ligeiramente diferentes para gerar uma proposta alternativa independente.

**Diferenças do Architect A:**
- Foco em **comunicação entre serviços** (event-driven vs request-driven)
- Abordagem **top-down** (começar pelo gateway e descer)
- Ênfase em **escalabilidade e resiliência**

### 7.1 Definição do Segundo Agente: Alternative Architect Agent

In [ ]:
# ============================================
# Agente 2: Alternative Architect Agent
# ============================================

alternative_architect_agent = Agent(
    role="Alternative Software Architect",
    goal="""Propor uma arquitetura de microsserviços ALTERNATIVA baseada em requisitos textuais.
    
    Sua abordagem é DIFERENTE do arquiteto tradicional:
    1. Comece pelo API Gateway e identifique os serviços de borda
    2. Pense em comunicação event-driven entre serviços
    3. Priorize escalabilidade e isolamento de falhas
    4. Considere serviços de infraestrutura (discovery, config, monitoring)
    
    Para cada microsserviço, você deve:
    1. Identificar um nome descritivo
    2. Listar as responsabilidades principais
    3. Especificar quais outros serviços ele se comunica""",
    
    backstory="""Você é um arquiteto de software sênior com visão inovadora.
    
    Sua experiência inclui:
    - Arquiteturas event-driven e message brokers
    - Sistemas distribuídos de alta escalabilidade
    - Microserviços com foco em resiliência (circuit breaker, retry)
    - Infraestrutura como código e service mesh
    
    Você frequentemente propõe soluções não convencionais que
    desafiam o status quo, sempre pensando em escalabilidade futura.""",
    
    llm=llm,
    verbose=True,
    memory=True,
)

print("✓ Alternative Architect Agent criado com sucesso")
print(f"  Role: {alternative_architect_agent.role}")
print(f"  Model: {google_model}")
print(f"  Abordagem: Top-down, event-driven, escalabilidade")

### 7.2 Definição da Task: Generate Alternative Architecture

In [ ]:
# ============================================
# Task 2: Generate Alternative Architecture
# ============================================

generate_alternative_task = Task(
    description="""Analyze the following system requirements and propose an ALTERNATIVE microservices architecture.
    
    Your approach should be DIFFERENT from a traditional DDD decomposition.
    Think about event-driven communication, API Gateway patterns, and scalability.
    
    Your output MUST be EXACTLY in this CSV format (no markdown, no explanations):
    
    Microservice,Responsibilities,Communicates With
    Service Name,responsibility1;responsibility2;responsibility3,Service1;Service2;Service3
    
    Rules:
    - One row per microservice
    - Separate multiple items with semicolons (;)
    - NO markdown formatting, NO headers, NO extra text
    - Include infrastructure services (gateway, config, discovery)
    
    System Requirements:
    {requirements}
    """,
    
    expected_output="CSV format with alternative microservices architecture",
    agent=alternative_architect_agent,
)

print("✓ Task 'Generate Alternative Architecture' criada com sucesso")
print(f"  Agent: {generate_alternative_task.agent.role}")
print(f"  Expected Output Format: CSV")

### 7.3 Executar Architect B com PetClinic

In [ ]:
# ============================================
# Executar Agente 2 (Alternative Architect) com PetClinic
# ============================================

print("="*60)
print("EXECUTANDO AGENTE 2: ALTERNATIVE ARCHITECT")
print("="*60)
print(f"\nSistema: Spring PetClinic")
print(f"Agente: {alternative_architect_agent.role}")
print(f"Task: Generate Alternative Architecture\n")

try:
    # Criar Crew com o agente alternativo
    crew_b = Crew(
        agents=[alternative_architect_agent],
        tasks=[generate_alternative_task],
        verbose=True,
    )
    
    # Executar
    petclinic_alternative_output = crew_b.kickoff(
        inputs={"requirements": petclinic_requirements}
    )
    
    print("\n" + "="*60)
    print("ARQUITETURA ALTERNATIVA GERADA")
    print("="*60)
    print(petclinic_alternative_output)
    print("\n✓ Agente alternativo executado com sucesso!")
    
except Exception as e:
    print(f"✗ Erro ao executar agente alternativo: {str(e)}")
    petclinic_alternative_output = None

---
## Fase 4: Agente Validador e Consolidação

O **Validator Agent** recebe as duas propostas (Architect A e Architect B) e:
1. Compara as similaridades e diferenças
2. Gera uma arquitetura consolidada (unindo o melhor de ambas)
3. Explica as decisões de design

### 8.1 Definição do Validador

In [ ]:
# ============================================
# Agente 3: Validator Agent
# ============================================

validator_agent = Agent(
    role="Architecture Validator",
    goal="""Comparar duas propostas de arquitetura de microsserviços e gerar uma versão consolidada.
    
    Você deve:
    1. Analisar ambas as propostas (Architect A e Architect B)
    2. Identificar serviços comuns entre elas
    3. Identificar serviços únicos de cada proposta
    4. Consolidar em uma arquitetura final que combine o melhor de ambas
    5. Explicar as decisões de consolidação""",
    
    backstory="""Você é um arquiteto líder com vasta experiência em revisão de designs.
    
    Sua especialidade:
    - Revisão e validação de arquiteturas de software
    - Identificação de overlaps e gaps em propostas
    - Tomada de decisão baseada em trade-offs
    - Comunicação clara de decisões arquiteturais
    
    Você consegue enxergar o melhor de diferentes abordagens
    e combiná-las em uma solução coesa e bem fundamentada.""",
    
    llm=llm,
    verbose=True,
    memory=True,
)

print("✓ Validator Agent criado com sucesso")
print(f"  Role: {validator_agent.role}")
print(f"  Model: {google_model}")

### 8.2 Definição da Task: Consolidate Architectures

In [ ]:
# ============================================
# Task 3: Consolidate Architectures
# ============================================

consolidate_task = Task(
    description="""Compare the two architecture proposals below and generate a consolidated version.
    
    ARCHITECTURE A (Traditional DDD approach):
    {architecture_a}
    
    ARCHITECTURE B (Alternative approach):
    {architecture_b}
    
    Your output MUST be in this format:
    
    === CONSOLIDATED ARCHITECTURE ===
    Microservice,Responsibilities,Communicates With,Source
    Service Name,resp1;resp2,ServiceX;ServiceY,A+B
    
    === DECISIONS ===
    - Service X: Kept from both proposals because...
    - Service Y: Merged from A's name and B's responsibilities because...
    - Service Z: Added from B only because...
    
    Rules:
    - Include a 'Source' column indicating where each service came from (A, B, or A+B)
    - Explain key decisions
    - Be specific about why certain services were kept, merged, or discarded""",
    
    expected_output="Consolidated architecture with decisions explained",
    agent=validator_agent,
)

print("✓ Task 'Consolidate Architectures' criada com sucesso")
print(f"  Agent: {consolidate_task.agent.role}")

### 8.3 Executar Validação e Consolidação

In [ ]:
# ============================================
# Executar Validador com as duas propostas
# ============================================

print("="*60)
print("EXECUTANDO VALIDADOR: CONSOLIDAÇÃO")
print("="*60)

if petclinic_architecture_output and petclinic_alternative_output:
    print("\n✓ Ambas as propostas disponíveis para comparação")
    print(f"  Proposta A (tradicional): {len(str(petclinic_architecture_output))} chars")
    print(f"  Proposta B (alternativa): {len(str(petclinic_alternative_output))} chars\n")
    
    try:
        # Criar Crew com o validador
        crew_validator = Crew(
            agents=[validator_agent],
            tasks=[consolidate_task],
            verbose=True,
        )
        
        # Executar consolidação
        consolidated_output = crew_validator.kickoff(
            inputs={
                "architecture_a": str(petclinic_architecture_output),
                "architecture_b": str(petclinic_alternative_output)
            }
        )
        
        print("\n" + "="*60)
        print("ARQUITETURA CONSOLIDADA")
        print("="*60)
        print(consolidated_output)
        print("\n✓ Consolidação executada com sucesso!")
        
    except Exception as e:
        print(f"✗ Erro na consolidação: {str(e)}")
        consolidated_output = None
else:
    print("⚠ Propostas não disponíveis. Execute as células das Fases 2 e 3 primeiro.")
    consolidated_output = None

---
## Fase 5: Avaliação com Referência (Precision, Recall, F1)

Comparamos a arquitetura consolidada (ou as propostas individuais) com a arquitetura de referência.

**Métricas:**
- **Precision:** Proporção de serviços gerados que estão na referência
- **Recall:** Proporção de serviços da referência que foram gerados
- **F1-Score:** Média harmônica entre Precision e Recall

### 9.1 Carregar Arquitetura de Referência (PetClinic)

In [ ]:
# ============================================
# Referência PetClinic (baseada na arquitetura oficial)
# ============================================

petclinic_reference_services = [
    "api-gateway",
    "config-server",
    "discovery-server",
    "admin-server",
    "customers-service",
    "vets-service",
    "visits-service",
]

print("✓ Referência PetClinic carregada")
print(f"  Serviços de referência ({len(petclinic_reference_services)}):")
for s in petclinic_reference_services:
    print(f"    - {s}")

### 9.2 Carregar Arquitetura de Referência (Bookstore)

In [ ]:
# ============================================
# Referência Bookstore
# ============================================

bookstore_reference_services = [
    "api-gateway",
    "config-server",
    "discovery-server",
    "book-service",
    "order-service",
    "customer-service",
    "inventory-service",
    "shipping-service",
    "notification-service",
]

print("✓ Referência Bookstore carregada")
print(f"  Serviços de referência ({len(bookstore_reference_services)}):")
for s in bookstore_reference_services:
    print(f"    - {s}")

### 9.3 Funções de Avaliação

In [ ]:
# ============================================
# Funções de Avaliação
# ============================================

def parse_csv_architecture(csv_text: str) -> List[str]:
    """Parse CSV output from agent into list of service names."""
    services = []
    try:
        lines = csv_text.strip().split('\n')
        for line in lines:
            line = line.strip()
            if not line or line.startswith('Microservice'):
                continue
            # CSV format: Service Name,Responsibilities,Communicates With
            parts = line.split(',')
            if len(parts) >= 1:
                service_name = parts[0].strip().lower()
                # Normalize: remove 'service' suffix for comparison
                service_name = service_name.replace(' ', '-')
                if service_name and not service_name.startswith('#'):
                    services.append(service_name)
    except Exception as e:
        print(f"  ⚠ Erro ao parsear CSV: {e}")
    return services


def normalize_service_name(name: str) -> str:
    """Normalize service name for comparison."""
    name = name.lower().strip()
    name = name.replace(' ', '-')
    name = name.replace('_', '-')
    # Remove common suffixes for matching
    name = name.removesuffix('-service')
    name = name.removesuffix('-microservice')
    name = name.removesuffix('-ms')
    return name


def calculate_metrics(generated_services: List[str], reference_services: List[str]) -> Dict:
    """Calculate Precision, Recall, and F1-Score."""
    # Normalize both lists
    gen_normalized = set(normalize_service_name(s) for s in generated_services)
    ref_normalized = set(normalize_service_name(s) for s in reference_services)
    
    # Find matches
    true_positives = gen_normalized & ref_normalized
    false_positives = gen_normalized - ref_normalized
    false_negatives = ref_normalized - gen_normalized
    
    # Calculate metrics
    precision = len(true_positives) / len(gen_normalized) if gen_normalized else 0.0
    recall = len(true_positives) / len(ref_normalized) if ref_normalized else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        "true_positives": sorted(true_positives),
        "false_positives": sorted(false_positives),
        "false_negatives": sorted(false_negatives),
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "generated_count": len(gen_normalized),
        "reference_count": len(ref_normalized),
        "match_count": len(true_positives),
    }


def print_evaluation_report(metrics: Dict, title: str = "Evaluation Report"):
    """Print a formatted evaluation report."""
    print(f"\n{'='*60}")
    print(f"{title}")
    print(f"{'='*60}")
    print(f"\n📊 Métricas:")
    print(f"  Precision: {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)")
    print(f"  Recall:    {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)")
    print(f"  F1-Score:  {metrics['f1_score']:.4f} ({metrics['f1_score']*100:.2f}%)")
    print(f"\n📋 Detalhes:")
    print(f"  Serviços Gerados:  {metrics['generated_count']}")
    print(f"  Serviços Referência: {metrics['reference_count']}")
    print(f"  Match (TP): {metrics['match_count']}")
    print(f"  False Positives: {len(metrics['false_positives'])}")
    print(f"  False Negatives: {len(metrics['false_negatives'])}")
    
    if metrics['false_positives']:
        print(f"\n  ❌ False Positives (gerados mas não na referência):")
        for s in metrics['false_positives']:
            print(f"     - {s}")
    
    if metrics['false_negatives']:
        print(f"\n  ⚠ False Negatives (na referência mas não gerados):")
        for s in metrics['false_negatives']:
            print(f"     - {s}")
    
    if metrics['true_positives']:
        print(f"\n  ✅ True Positives (match):")
        for s in metrics['true_positives']:
            print(f"     - {s}")


print("✓ Funções de avaliação carregadas com sucesso")

### 9.4 Avaliar Proposta A (Architect Tradicional)

In [ ]:
# ============================================
# Avaliar Proposta A vs Referência PetClinic
# ============================================

if petclinic_architecture_output:
    print("Avaliando Proposta A (Software Architect)...\n")
    
    # Parsear CSV
    services_a = parse_csv_architecture(str(petclinic_architecture_output))
    print(f"Serviços extraídos da Proposta A: {services_a}")
    
    # Calcular métricas
    metrics_a = calculate_metrics(services_a, petclinic_reference_services)
    
    # Exibir relatório
    print_evaluation_report(metrics_a, "AVALIAÇÃO - PROPOSTA A (Software Architect)")
else:
    print("⚠ Proposta A não disponível. Execute a Fase 2 primeiro.")
    metrics_a = None

### 9.5 Avaliar Proposta B (Alternative Architect)

In [ ]:
# ============================================
# Avaliar Proposta B vs Referência PetClinic
# ============================================

if petclinic_alternative_output:
    print("Avaliando Proposta B (Alternative Architect)...\n")
    
    # Parsear CSV
    services_b = parse_csv_architecture(str(petclinic_alternative_output))
    print(f"Serviços extraídos da Proposta B: {services_b}")
    
    # Calcular métricas
    metrics_b = calculate_metrics(services_b, petclinic_reference_services)
    
    # Exibir relatório
    print_evaluation_report(metrics_b, "AVALIAÇÃO - PROPOSTA B (Alternative Architect)")
else:
    print("⚠ Proposta B não disponível. Execute a Fase 3 primeiro.")
    metrics_b = None

### 9.6 Avaliar Arquitetura Consolidada

In [ ]:
# ============================================
# Avaliar Arquitetura Consolidada vs Referência
# ============================================

if consolidated_output:
    print("Avaliando Arquitetura Consolidada...\n")
    
    # Parsear CSV da consolidada
    services_consolidated = parse_csv_architecture(str(consolidated_output))
    print(f"Serviços extraídos da Consolidada: {services_consolidated}")
    
    # Calcular métricas
    metrics_consolidated = calculate_metrics(services_consolidated, petclinic_reference_services)
    
    # Exibir relatório
    print_evaluation_report(metrics_consolidated, "AVALIAÇÃO - ARQUITETURA CONSOLIDADA")
else:
    print("⚠ Arquitetura Consolidada não disponível. Execute a Fase 4 primeiro.")
    metrics_consolidated = None

### 9.7 Comparação Final

In [ ]:
# ============================================
# Tabela Comparativa
# ============================================

print("\n" + "█"*70)
print("COMPARAÇÃO FINAL - PETCLINIC")
print("█"*70 + "\n")

print(f"{'Métrica':<20} {'Proposta A':<15} {'Proposta B':<15} {'Consolidada':<15}")
print(f"{'-'*20} {'-'*15} {'-'*15} {'-'*15}")

if metrics_a and metrics_b and metrics_consolidated:
    print(f"{'Precision':<20} {metrics_a['precision']:<15.4f} {metrics_b['precision']:<15.4f} {metrics_consolidated['precision']:<15.4f}")
    print(f"{'Recall':<20} {metrics_a['recall']:<15.4f} {metrics_b['recall']:<15.4f} {metrics_consolidated['recall']:<15.4f}")
    print(f"{'F1-Score':<20} {metrics_a['f1_score']:<15.4f} {metrics_b['f1_score']:<15.4f} {metrics_consolidated['f1_score']:<15.4f}")
    print(f"{'Serviços':<20} {metrics_a['generated_count']:<15} {metrics_b['generated_count']:<15} {metrics_consolidated['generated_count']:<15}")
    print(f"{'Match (TP)':<20} {metrics_a['match_count']:<15} {metrics_b['match_count']:<15} {metrics_consolidated['match_count']:<15}")
    
    # Determinar melhor proposta
    scores = {
        'Proposta A': metrics_a['f1_score'],
        'Proposta B': metrics_b['f1_score'],
        'Consolidada': metrics_consolidated['f1_score'],
    }
    best = max(scores, key=scores.get)
    print(f"\n🏆 Melhor proposta: {best} (F1 = {scores[best]:.4f})")
else:
    print("Métricas insuficientes para comparação.")

print("\n" + "█"*70)

---
## Fase 6: Experimento Completo com Bookstore

Executamos o experimento completo com o sistema Bookstore para validar a abordagem em outro domínio.

### 10.1 Requisitos do Bookstore

In [ ]:
# ============================================
# Requisitos do Bookstore
# ============================================

bookstore_requirements = """
Bookstore Microservices - Requirements

1. Book Catalog Management:
   - Add, update, and delete books from the catalog.
   - Search books by title, author, genre, or ISBN.
   - View book details including price, stock, and description.

2. Customer Management:
   - Register new customers with personal information.
   - Update customer profiles and preferences.
   - View customer purchase history.

3. Order Management:
   - Create new orders with multiple books.
   - Track order status (pending, confirmed, shipped, delivered).
   - Cancel orders before shipment.

4. Inventory Management:
   - Track stock levels for each book.
   - Receive low-stock alerts.
   - Update inventory when orders are placed or received.

5. Shipping Management:
   - Manage shipping addresses and methods.
   - Track shipment status and delivery dates.
   - Calculate shipping costs based on weight and destination.

6. Notification Service:
   - Send order confirmation emails to customers.
   - Send shipment updates and delivery notifications.
   - Send promotional offers and newsletters.

7. API Gateway:
   - Route client requests to appropriate backend services.
   - Handle authentication and rate limiting.

8. Service Discovery:
   - Register and locate services dynamically.

9. Centralized Configuration:
   - Manage service configurations centrally.
"""

print("✓ Requisitos Bookstore carregados")
print(f"  Tamanho: {len(bookstore_requirements)} caracteres")
print(f"  9 áreas de responsabilidade identificadas")

### 10.2 Executar Architect A com Bookstore

In [ ]:
# ============================================
# Executar Agente 1 com Bookstore
# ============================================

print("="*60)
print("EXECUTANDO AGENTE 1 COM BOOKSTORE")
print("="*60)

try:
    crew_bookstore_a = Crew(
        agents=[software_architect_agent],
        tasks=[generate_architecture_task],
        verbose=True,
    )
    
    bookstore_architecture_a = crew_bookstore_a.kickoff(
        inputs={"requirements": bookstore_requirements}
    )
    
    print("\n" + "="*60)
    print("ARQUITETURA BOOKSTORE - PROPOSTA A")
    print("="*60)
    print(bookstore_architecture_a)
    print("\n✓ Proposta A para Bookstore gerada com sucesso!")
    
except Exception as e:
    print(f"✗ Erro: {str(e)}")
    bookstore_architecture_a = None

### 10.3 Executar Architect B com Bookstore

In [ ]:
# ============================================
# Executar Agente 2 com Bookstore
# ============================================

print("="*60)
print("EXECUTANDO AGENTE 2 COM BOOKSTORE")
print("="*60)

try:
    crew_bookstore_b = Crew(
        agents=[alternative_architect_agent],
        tasks=[generate_alternative_task],
        verbose=True,
    )
    
    bookstore_architecture_b = crew_bookstore_b.kickoff(
        inputs={"requirements": bookstore_requirements}
    )
    
    print("\n" + "="*60)
    print("ARQUITETURA BOOKSTORE - PROPOSTA B")
    print("="*60)
    print(bookstore_architecture_b)
    print("\n✓ Proposta B para Bookstore gerada com sucesso!")
    
except Exception as e:
    print(f"✗ Erro: {str(e)}")
    bookstore_architecture_b = None

### 10.4 Consolidar Arquiteturas do Bookstore

In [ ]:
# ============================================
# Consolidar Bookstore
# ============================================

print("="*60)
print("CONSOLIDANDO ARQUITETURAS DO BOOKSTORE")
print("="*60)

if bookstore_architecture_a and bookstore_architecture_b:
    try:
        crew_bookstore_validator = Crew(
            agents=[validator_agent],
            tasks=[consolidate_task],
            verbose=True,
        )
        
        bookstore_consolidated = crew_bookstore_validator.kickoff(
            inputs={
                "architecture_a": str(bookstore_architecture_a),
                "architecture_b": str(bookstore_architecture_b)
            }
        )
        
        print("\n" + "="*60)
        print("ARQUITETURA CONSOLIDADA - BOOKSTORE")
        print("="*60)
        print(bookstore_consolidated)
        print("\n✓ Consolidação Bookstore concluída!")
        
    except Exception as e:
        print(f"✗ Erro na consolidação: {str(e)}")
        bookstore_consolidated = None
else:
    print("⚠ Propostas não disponíveis. Execute as células 10.2 e 10.3 primeiro.")
    bookstore_consolidated = None

### 10.5 Avaliar Bookstore

In [ ]:
# ============================================
# Avaliar Bookstore
# ============================================

print("\n" + "█"*70)
print("AVALIAÇÃO BOOKSTORE")
print("█"*70 + "\n")

# Avaliar Proposta A
if bookstore_architecture_a:
    services_bs_a = parse_csv_architecture(str(bookstore_architecture_a))
    metrics_bs_a = calculate_metrics(services_bs_a, bookstore_reference_services)
    print_evaluation_report(metrics_bs_a, "BOOKSTORE - PROPOSTA A")
else:
    metrics_bs_a = None

# Avaliar Proposta B
if bookstore_architecture_b:
    services_bs_b = parse_csv_architecture(str(bookstore_architecture_b))
    metrics_bs_b = calculate_metrics(services_bs_b, bookstore_reference_services)
    print_evaluation_report(metrics_bs_b, "BOOKSTORE - PROPOSTA B")
else:
    metrics_bs_b = None

# Avaliar Consolidada
if bookstore_consolidated:
    services_bs_c = parse_csv_architecture(str(bookstore_consolidated))
    metrics_bs_c = calculate_metrics(services_bs_c, bookstore_reference_services)
    print_evaluation_report(metrics_bs_c, "BOOKSTORE - CONSOLIDADA")
else:
    metrics_bs_c = None

### 10.6 Comparação Final Bookstore

In [ ]:
# ============================================
# Comparação Final Bookstore
# ============================================

print("\n" + "█"*70)
print("COMPARAÇÃO FINAL - BOOKSTORE")
print("█"*70 + "\n")

if metrics_bs_a and metrics_bs_b and metrics_bs_c:
    print(f"{'Métrica':<20} {'Proposta A':<15} {'Proposta B':<15} {'Consolidada':<15}")
    print(f"{'-'*20} {'-'*15} {'-'*15} {'-'*15}")
    print(f"{'Precision':<20} {metrics_bs_a['precision']:<15.4f} {metrics_bs_b['precision']:<15.4f} {metrics_bs_c['precision']:<15.4f}")
    print(f"{'Recall':<20} {metrics_bs_a['recall']:<15.4f} {metrics_bs_b['recall']:<15.4f} {metrics_bs_c['recall']:<15.4f}")
    print(f"{'F1-Score':<20} {metrics_bs_a['f1_score']:<15.4f} {metrics_bs_b['f1_score']:<15.4f} {metrics_bs_c['f1_score']:<15.4f}")
    print(f"{'Serviços':<20} {metrics_bs_a['generated_count']:<15} {metrics_bs_b['generated_count']:<15} {metrics_bs_c['generated_count']:<15}")
    print(f"{'Match (TP)':<20} {metrics_bs_a['match_count']:<15} {metrics_bs_b['match_count']:<15} {metrics_bs_c['match_count']:<15}")
    
    scores_bs = {
        'Proposta A': metrics_bs_a['f1_score'],
        'Proposta B': metrics_bs_b['f1_score'],
        'Consolidada': metrics_bs_c['f1_score'],
    }
    best_bs = max(scores_bs, key=scores_bs.get)
    print(f"\n🏆 Melhor proposta para Bookstore: {best_bs} (F1 = {scores_bs[best_bs]:.4f})")
else:
    print("Métricas insuficientes para comparação.")

print("\n" + "█"*70)

---
## Conclusão

### Resumo dos Resultados

| Sistema | Proposta A (F1) | Proposta B (F1) | Consolidada (F1) |
|---------|-----------------|-----------------|------------------|
| PetClinic | - | - | - |
| Bookstore | - | - | - |

### Observações
- A abordagem multiagente permite explorar diferentes perspectivas arquiteturais
- O validador combina o melhor de cada proposta
- A consolidação tende a melhorar o Recall (cobre mais serviços da referência)
- Precision pode ser afetada por serviços extras propostos pelos agentes

### Trabalhos Futuros
- Adicionar mais agentes com especializações diferentes
- Implementar feedback loop entre validador e arquitetos
- Avaliar com mais sistemas de referência
- Incorporar análise de qualidade arquitetural (coesão, acoplamento)

In [ ]:
# ============================================
# Resumo Final do Experimento
# ============================================

print("\n" + "█"*70)
print("EXPERIMENTO MULTIAGENTE - RESUMO FINAL")
print("█"*70 + "\n")

print("📋 FASES EXECUTADAS:")
print("  ✅ Fase 1: Setup & Validação do Ambiente")
print("  ✅ Fase 2: Primeiro Agente (Software Architect)")
print("  ✅ Fase 3: Segundo Agente (Alternative Architect)")
print("  ✅ Fase 4: Validador e Consolidação")
print("  ✅ Fase 5: Avaliação (Precision, Recall, F1)")
print("  ✅ Fase 6: Experimento Completo (PetClinic + Bookstore)\n")

print("🤖 AGENTES CRIADOS:")
print(f"  1. {software_architect_agent.role} - Abordagem DDD tradicional")
print(f"  2. {alternative_architect_agent.role} - Abordagem event-driven/top-down")
print(f"  3. {validator_agent.role} - Comparação e consolidação\n")

print("📊 MÉTRICAS DE AVALIAÇÃO:")
print("  - Precision: proporção de serviços corretos entre os gerados")
print("  - Recall: proporção de serviços da referência que foram gerados")
print("  - F1-Score: média harmônica entre Precision e Recall\n")

print("🔧 TECNOLOGIAS UTILIZADAS:")
print(f"  - Modelo: {google_model} (Google Gemini)")
print("  - Framework: LangChain + CrewAI")
print("  - Formato de saída: CSV para parsing automático\n")

print("💡 PRÓXIMOS PASSOS:")
print("  1. Analisar os resultados das tabelas comparativas")
print("  2. Identificar padrões de erros (false positives/negatives)")
print("  3. Refinar prompts dos agentes com base nos resultados")
print("  4. Expandir para mais sistemas de referência\n")

print("█"*70)